# Tabela `fact_locacao` (silver)

Carrega `rental` do bronze (`nessie.bronze_sakila.rental`), remove
duplicatas, renomeia as colunas para o padrao em portugues e escreve como
`fact_locacao` no schema `silver`. A escrita é
idempotente (`createOrReplace`), então reexecuções recriam a tabela sem
duplicar dados.

In [ ]:
SPARK_APP_NAME = "cria-fact-rental-silver"

from nucleo import cria_spark_session

spark = cria_spark_session(SPARK_APP_NAME)
spark

In [ ]:
TABELA_ORIGEM = "nessie.bronze_sakila.rental"
TABELA_DESTINO = "nessie.silver.fact_locacao"

df = (
    spark.table(TABELA_ORIGEM)
    .dropDuplicates()
    .withColumnRenamed("rental_id", "locacao_id")
    .withColumnRenamed("rental_date", "data_locacao")
    .withColumnRenamed("inventory_id", "inventario_id")
    .withColumnRenamed("customer_id", "cliente_id")
    .withColumnRenamed("staff_id", "funcionario_id")
    .withColumnRenamed("return_date", "data_retorno")
    .withColumnRenamed("last_update", "ultima_atualizacao")
)
df.writeTo(TABELA_DESTINO).createOrReplace()

In [ ]:
n = spark.table(TABELA_DESTINO).count()
print(f"{TABELA_DESTINO}: {n} linhas")